In [1]:
from pathlib import Path

import pandas as pd

In [2]:
# Racine du projet
PROJECT_DIR = Path.cwd().parent

# Dossiers du projet
RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
REPORTS_DIR = PROJECT_DIR / "reports"

# Création automatique des dossiers de sortie
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print("Dossier des données brutes :", RAW_DIR)
print("Dossier des données nettoyées :", PROCESSED_DIR)
print("Dossier des rapports :", REPORTS_DIR)

Dossier des données brutes : C:\Users\abide\projet_lerouge_moulin\data\raw
Dossier des données nettoyées : C:\Users\abide\projet_lerouge_moulin\data\processed
Dossier des rapports : C:\Users\abide\projet_lerouge_moulin\reports


In [3]:
import xml.etree.ElementTree as ET


def lire_xml(chemin_fichier, balise_element):
    """
    Lit un fichier XML et retourne un DataFrame pandas.

    Paramètres
    ----------
    chemin_fichier : Path
        Chemin du fichier XML.
    balise_element : str
        Nom de la balise contenant chaque enregistrement.
        Exemple : "Client" ou "Product".
    """

    arbre = ET.parse(chemin_fichier)
    racine = arbre.getroot()

    donnees = []

    for element in racine.findall(balise_element):
        ligne = {}

        for champ in element:
            ligne[champ.tag] = champ.text

        donnees.append(ligne)

    return pd.DataFrame(donnees)

In [4]:
CLIENT_FILE = RAW_DIR / "client.xml"
PRODUCT_CSV_FILE = RAW_DIR / "product.csv"
PRODUCT_XML_FILE = RAW_DIR / "product.xml"
TRANSACTIONS_FILE = RAW_DIR / "transactions_2026.csv"

In [5]:
clients = lire_xml(
    chemin_fichier=CLIENT_FILE,
    balise_element="Client"
)

products_csv = pd.read_csv(PRODUCT_CSV_FILE)

products_xml = lire_xml(
    chemin_fichier=PRODUCT_XML_FILE,
    balise_element="Product"
)

transactions = pd.read_csv(TRANSACTIONS_FILE)

In [6]:
print("Nombre de clients :", len(clients))
print("Nombre de produits CSV :", len(products_csv))
print("Nombre de produits XML :", len(products_xml))
print("Nombre de transactions :", len(transactions))

Nombre de clients : 435
Nombre de produits CSV : 180
Nombre de produits XML : 20
Nombre de transactions : 10343


In [7]:
display(clients.head())
display(products_csv.head())
display(products_xml.head())
display(transactions.head())

,client_id,name,age,sexe,opening_date,status
0,CLI0001,Julie Durand,81,M,2023-03-26,active
1,CLI0002,Laura Bernard,58,M,2022-10-30,active
2,CLI0003,Thomas Moreau,18,M,2025-03-07,active
3,CLI0004,Sophie Moreau,64,M,2025-08-03,active
4,CLI0005,Jean Petit,52,F,2021-06-20,active


,product_id,product_name,product_price,price_date
0,PRD0001,Peinture 125,260.19,2026-07-08
1,PRD0002,Marteau 854,38.12,2026-07-24
2,PRD0003,Peinture 704,149.12,2026-07-01
3,PRD0004,Vis 338,178.11,2026-07-01
4,PRD0005,Vis 833,228.33,2026-07-18


,product_id,product_name,product_price,price_date
0,PRD0181,Tournevis 561,318.21,2026-07-10
1,PRD0182,Tournevis 412,200.1,2026-07-02
2,PRD0183,Peinture 881,74.71,2026-07-07
3,PRD0184,Peinture 260,85.85,2026-07-18
4,PRD0185,Marteau 102,144.46,2026-07-23


,trsx_id,client_id,cart_id,product_id,date,amount,price,products_price,cart_price
0,TRX000001,CLI0349,CRT00519,PRD0152,2026-04-12 07:24:58,10,336.37,3363.70,4507.13
1,TRX000002,CLI0335,CRT00992,PRD0050,2026-03-31 01:20:13,5,164.79,823.95,1606.03
2,TRX000003,CLI0246,CRT00421,PRD0176,2026-04-29 16:12:53,8,288.19,2305.52,6573.80
3,TRX000004,CLI0249,CRT01364,PRD0059,2026-01-14 15:52:09,2,252.15,504.30,5839.25
4,TRX000005,CLI0057,CRT02032,PRD0050,2026-07-07 02:10:57,6,164.79,988.74,988.74


In [8]:
print("Colonnes CLIENT :", clients.columns.tolist())
print("Colonnes PRODUCT CSV :", products_csv.columns.tolist())
print("Colonnes PRODUCT XML :", products_xml.columns.tolist())
print("Colonnes TRANSACTIONS :", transactions.columns.tolist())

Colonnes CLIENT : ['client_id', 'name', 'age', 'sexe', 'opening_date', 'status']
Colonnes PRODUCT CSV : ['product_id', 'product_name', 'product_price', 'price_date']
Colonnes PRODUCT XML : ['product_id', 'product_name', 'product_price', 'price_date']
Colonnes TRANSACTIONS : ['trsx_id', 'client_id', 'cart_id', 'product_id', 'date', 'amount', 'price', 'products_price', 'cart_price']


In [9]:
print("Types CLIENT")
print(clients.dtypes)

print("\nTypes PRODUCT CSV")
print(products_csv.dtypes)

print("\nTypes PRODUCT XML")
print(products_xml.dtypes)

print("\nTypes TRANSACTIONS")
print(transactions.dtypes)

Types CLIENT
client_id       object
name            object
age             object
sexe            object
opening_date    object
status          object
dtype: object

Types PRODUCT CSV
product_id        object
product_name      object
product_price    float64
price_date        object
dtype: object

Types PRODUCT XML
product_id       object
product_name     object
product_price    object
price_date       object
dtype: object

Types TRANSACTIONS
trsx_id            object
client_id          object
cart_id            object
product_id         object
date               object
amount              int64
price             float64
products_price    float64
cart_price        float64
dtype: object


In [10]:
clients["age"] = pd.to_numeric(
    clients["age"],
    errors="coerce"
).astype("Int64")

clients["opening_date"] = pd.to_datetime(
    clients["opening_date"],
    errors="coerce"
)

clients["client_id"] = clients["client_id"].astype("string")
clients["name"] = clients["name"].astype("string")
clients["sexe"] = clients["sexe"].astype("string")
clients["status"] = clients["status"].astype("string")

In [11]:
for dataframe in [products_csv, products_xml]:

    dataframe["product_id"] = dataframe["product_id"].astype("string")
    dataframe["product_name"] = dataframe["product_name"].astype("string")

    dataframe["product_price"] = pd.to_numeric(
        dataframe["product_price"],
        errors="coerce"
    )

    dataframe["price_date"] = pd.to_datetime(
        dataframe["price_date"],
        errors="coerce"
    )

In [12]:
products_all = pd.concat(
    [products_csv, products_xml],
    ignore_index=True
)

In [13]:
print("Nombre total de produits :", len(products_all))
print(
    "Nombre d'identifiants produit uniques :",
    products_all["product_id"].nunique()
)

display(products_all.head())
display(products_all.tail())

Nombre total de produits : 200
Nombre d'identifiants produit uniques : 200


,product_id,product_name,product_price,price_date
0,PRD0001,Peinture 125,260.19,2026-07-08
1,PRD0002,Marteau 854,38.12,2026-07-24
2,PRD0003,Peinture 704,149.12,2026-07-01
3,PRD0004,Vis 338,178.11,2026-07-01
4,PRD0005,Vis 833,228.33,2026-07-18


,product_id,product_name,product_price,price_date
195,PRD0196,Ampoule 121,349.85,2026-07-22
196,PRD0197,Ampoule 141,267.68,2026-07-06
197,PRD0198,Pince 767,156.17,2026-07-09
198,PRD0199,Ampoule 546,223.09,2026-07-16
199,PRD0200,Scie 456,144.40,2026-07-11


In [14]:
transactions["trsx_id"] = transactions["trsx_id"].astype("string")
transactions["client_id"] = transactions["client_id"].astype("string")
transactions["cart_id"] = transactions["cart_id"].astype("string")
transactions["product_id"] = transactions["product_id"].astype("string")

transactions["date"] = pd.to_datetime(
    transactions["date"],
    errors="coerce"
)

transactions["amount"] = pd.to_numeric(
    transactions["amount"],
    errors="coerce"
).astype("Int64")

for colonne in [
    "price",
    "products_price",
    "cart_price"
]:
    transactions[colonne] = pd.to_numeric(
        transactions[colonne],
        errors="coerce"
    )

In [15]:
print("Types CLIENT après conversion")
print(clients.dtypes)

print("\nTypes PRODUIT après conversion")
print(products_all.dtypes)

print("\nTypes TRANSACTIONS après conversion")
print(transactions.dtypes)

Types CLIENT après conversion
client_id       string[python]
name            string[python]
age                      Int64
sexe            string[python]
opening_date    datetime64[ns]
status          string[python]
dtype: object

Types PRODUIT après conversion
product_id       string[python]
product_name     string[python]
product_price           float64
price_date       datetime64[ns]
dtype: object

Types TRANSACTIONS après conversion
trsx_id           string[python]
client_id         string[python]
cart_id           string[python]
product_id        string[python]
date              datetime64[ns]
amount                     Int64
price                    float64
products_price           float64
cart_price               float64
dtype: object


In [16]:
def verifier_valeurs_manquantes(dataframe, nom_table):
    """
    Retourne le nombre de valeurs manquantes par colonne.
    """

    resultat = dataframe.isna().sum()

    resultat = resultat[resultat > 0]

    if resultat.empty:
        print(f"{nom_table} : aucune valeur manquante.")
    else:
        print(f"{nom_table} : valeurs manquantes détectées")
        print(resultat)

    return resultat

In [17]:
manquants_clients = verifier_valeurs_manquantes(
    clients,
    "CLIENT"
)

manquants_produits = verifier_valeurs_manquantes(
    products_all,
    "PRODUIT"
)

manquants_transactions = verifier_valeurs_manquantes(
    transactions,
    "TRANSACTIONS"
)

CLIENT : aucune valeur manquante.
PRODUIT : aucune valeur manquante.
TRANSACTIONS : aucune valeur manquante.


In [18]:
def verifier_doublons_complets(dataframe, nom_table):
    """
    Détecte les lignes entièrement identiques.
    """

    masque_doublons = dataframe.duplicated(
        keep=False
    )

    doublons = dataframe[masque_doublons]

    if doublons.empty:
        print(f"{nom_table} : aucun doublon complet.")
    else:
        print(
            f"{nom_table} : "
            f"{len(doublons)} lignes impliquées dans des doublons complets."
        )

    return doublons

In [19]:
doublons_clients = verifier_doublons_complets(
    clients,
    "CLIENT"
)

doublons_produits = verifier_doublons_complets(
    products_all,
    "PRODUIT"
)

doublons_transactions = verifier_doublons_complets(
    transactions,
    "TRANSACTIONS"
)

CLIENT : aucun doublon complet.
PRODUIT : aucun doublon complet.
TRANSACTIONS : aucun doublon complet.


In [20]:
def verifier_cle_unique(dataframe, colonne, nom_table):
    """
    Vérifie qu'une colonne destinée à être une clé primaire
    ne contient ni doublon ni valeur manquante.
    """

    valeurs_manquantes = dataframe[colonne].isna().sum()

    doublons = dataframe[
        dataframe[colonne].duplicated(keep=False)
    ].sort_values(colonne)

    cle_valide = (
        valeurs_manquantes == 0
        and doublons.empty
    )

    if cle_valide:
        print(
            f"{nom_table}.{colonne} : "
            "clé unique et complète."
        )
    else:
        print(
            f"{nom_table}.{colonne} : "
            "clé invalide."
        )

        print(
            "Valeurs manquantes :",
            valeurs_manquantes
        )

        print(
            "Lignes avec clé dupliquée :",
            len(doublons)
        )

    return {
        "table": nom_table,
        "colonne": colonne,
        "valeurs_manquantes": valeurs_manquantes,
        "doublons": len(doublons),
        "valide": cle_valide
    }, doublons

In [21]:
controle_client_id, doublons_client_id = verifier_cle_unique(
    clients,
    "client_id",
    "CLIENT"
)

controle_product_id, doublons_product_id = verifier_cle_unique(
    products_all,
    "product_id",
    "PRODUIT"
)

controle_trsx_id, doublons_trsx_id = verifier_cle_unique(
    transactions,
    "trsx_id",
    "TRANSACTIONS"
)

CLIENT.client_id : clé unique et complète.
PRODUIT.product_id : clé unique et complète.
TRANSACTIONS.trsx_id : clé unique et complète.


In [22]:
def verifier_integrite_referentielle(
    dataframe_source,
    colonne_source,
    dataframe_reference,
    colonne_reference,
    nom_controle
):
    """
    Vérifie que toutes les valeurs d'une clé étrangère
    existent dans la table de référence.
    """

    valeurs_reference = set(
        dataframe_reference[colonne_reference].dropna()
    )

    masque_inconnu = (
        dataframe_source[colonne_source].notna()
        & ~dataframe_source[colonne_source].isin(
            valeurs_reference
        )
    )

    lignes_inconnues = dataframe_source[
        masque_inconnu
    ].copy()

    valeurs_inconnues = sorted(
        lignes_inconnues[colonne_source]
        .dropna()
        .unique()
        .tolist()
    )

    if lignes_inconnues.empty:
        print(
            f"{nom_controle} : "
            "intégrité référentielle respectée."
        )
    else:
        print(
            f"{nom_controle} : "
            f"{len(lignes_inconnues)} ligne(s) invalide(s)."
        )

        print(
            "Valeurs inconnues :",
            valeurs_inconnues
        )

    return lignes_inconnues

In [23]:
transactions_clients_inconnus = verifier_integrite_referentielle(
    dataframe_source=transactions,
    colonne_source="client_id",
    dataframe_reference=clients,
    colonne_reference="client_id",
    nom_controle="TRANSACTIONS → CLIENT"
)

TRANSACTIONS → CLIENT : intégrité référentielle respectée.


In [24]:
transactions_produits_inconnus = verifier_integrite_referentielle(
    dataframe_source=transactions,
    colonne_source="product_id",
    dataframe_reference=products_all,
    colonne_reference="product_id",
    nom_controle="TRANSACTIONS → PRODUIT"
)

TRANSACTIONS → PRODUIT : intégrité référentielle respectée.


In [25]:
transactions_panier_produit_repetes = transactions[
    transactions.duplicated(
        subset=["cart_id", "product_id"],
        keep=False
    )
].sort_values(
    ["cart_id", "product_id", "trsx_id"]
)

nombre_lignes_repetees = len(
    transactions_panier_produit_repetes
)

nombre_couples_repetes = (
    transactions_panier_produit_repetes[
        ["cart_id", "product_id"]
    ]
    .drop_duplicates()
    .shape[0]
)

print(
    "Nombre de couples (cart_id, product_id) répétés :",
    nombre_couples_repetes
)

print(
    "Nombre de lignes concernées :",
    nombre_lignes_repetees
)

display(
    transactions_panier_produit_repetes.head(10)
)

Nombre de couples (cart_id, product_id) répétés : 67
Nombre de lignes concernées : 134


,trsx_id,client_id,cart_id,product_id,date,amount,price,products_price,cart_price
3737,TRX003738,CLI0421,CRT00073,PRD0158,2026-01-15 18:45:37,10,12.47,124.70,3415.43
4861,TRX004862,CLI0421,CRT00073,PRD0158,2026-01-15 18:45:37,5,12.47,62.35,3415.43
1997,TRX001998,CLI0146,CRT00089,PRD0027,2026-01-08 14:04:27,5,209.57,1047.85,3674.03
10214,TRX010215,CLI0146,CRT00089,PRD0027,2026-01-08 14:04:27,2,209.57,419.14,3674.03
1807,TRX001808,CLI0272,CRT00139,PRD0048,2026-03-18 15:02:26,5,275.16,1375.80,3301.92
4071,TRX004072,CLI0272,CRT00139,PRD0048,2026-03-18 15:02:26,7,275.16,1926.12,3301.92
3038,TRX003039,CLI0247,CRT00235,PRD0174,2026-03-30 23:21:10,1,6.76,6.76,3583.37
6197,TRX006198,CLI0247,CRT00235,PRD0174,2026-03-30 23:21:10,5,6.76,33.80,3583.37
6074,TRX006075,CLI0162,CRT00376,PRD0039,2026-01-05 14:10:52,6,305.67,1834.02,5017.05
9649,TRX009650,CLI0162,CRT00376,PRD0039,2026-01-05 14:10:52,5,305.67,1528.35,5017.05


### Les répétitions du couple (cart_id, product_id) ne constituent
### pas des doublons complets. Elles correspondent à des lignes
### transactionnelles distinctes identifiées par des trsx_id uniques.

In [26]:
print("\n===== SYNTHÈSE DES CONTRÔLES =====")

print(
    "Valeurs manquantes :",
    (
        len(manquants_clients)
        + len(manquants_produits)
        + len(manquants_transactions)
    )
)

print(
    "Doublons complets :",
    (
        len(doublons_clients)
        + len(doublons_produits)
        + len(doublons_transactions)
    )
)

print(
    "Clients inconnus dans les transactions :",
    len(transactions_clients_inconnus)
)

print(
    "Produits inconnus dans les transactions :",
    len(transactions_produits_inconnus)
)

print(
    "Clé client_id valide :",
    controle_client_id["valide"]
)

print(
    "Clé product_id valide :",
    controle_product_id["valide"]
)

print(
    "Clé trsx_id valide :",
    controle_trsx_id["valide"]
)


===== SYNTHÈSE DES CONTRÔLES =====
Valeurs manquantes : 0
Doublons complets : 0
Clients inconnus dans les transactions : 0
Produits inconnus dans les transactions : 0
Clé client_id valide : True
Clé product_id valide : True
Clé trsx_id valide : True


In [27]:
clients_par_panier = (
    transactions
    .groupby("cart_id")["client_id"]
    .nunique()
)

paniers_plusieurs_clients = clients_par_panier[
    clients_par_panier > 1
]

if paniers_plusieurs_clients.empty:
    print("Tous les paniers sont associés à un seul client.")
else:
    print(
        "Nombre de paniers associés à plusieurs clients :",
        len(paniers_plusieurs_clients)
    )

    display(
        transactions[
            transactions["cart_id"].isin(
                paniers_plusieurs_clients.index
            )
        ].sort_values(["cart_id", "client_id"])
    )

Tous les paniers sont associés à un seul client.


In [28]:
dates_par_panier = (
    transactions
    .groupby("cart_id")["date"]
    .nunique()
)

paniers_plusieurs_dates = dates_par_panier[
    dates_par_panier > 1
]

if paniers_plusieurs_dates.empty:
    print("Tous les paniers possèdent une seule date.")
else:
    print(
        "Nombre de paniers associés à plusieurs dates :",
        len(paniers_plusieurs_dates)
    )

    display(
        transactions[
            transactions["cart_id"].isin(
                paniers_plusieurs_dates.index
            )
        ].sort_values(["cart_id", "date"])
    )

Tous les paniers possèdent une seule date.


In [29]:
transactions_quantite_invalide = transactions[
    transactions["amount"].isna()
    | (transactions["amount"] <= 0)
].copy()

if transactions_quantite_invalide.empty:
    print("Toutes les quantités sont strictement positives.")
else:
    print(
        "Nombre de transactions avec une quantité invalide :",
        len(transactions_quantite_invalide)
    )

    display(transactions_quantite_invalide)

Toutes les quantités sont strictement positives.


In [30]:
transactions_prix_invalide = transactions[
    transactions["price"].isna()
    | (transactions["price"] < 0)
].copy()

if transactions_prix_invalide.empty:
    print("Tous les prix unitaires sont valides.")
else:
    print(
        "Nombre de transactions avec un prix unitaire invalide :",
        len(transactions_prix_invalide)
    )

    display(transactions_prix_invalide)

Tous les prix unitaires sont valides.


In [31]:
import numpy as np

transactions["products_price_calcule"] = (
    transactions["amount"].astype("float64")
    * transactions["price"]
).round(2)

masque_products_price_incoherent = ~np.isclose(
    transactions["products_price"],
    transactions["products_price_calcule"],
    atol=0.01,
    equal_nan=False
)

transactions_products_price_incoherent = transactions[
    masque_products_price_incoherent
].copy()

if transactions_products_price_incoherent.empty:
    print(
        "Tous les montants de ligne respectent : "
        "products_price = amount × price."
    )
else:
    print(
        "Nombre de lignes avec un montant incohérent :",
        len(transactions_products_price_incoherent)
    )

    display(
        transactions_products_price_incoherent[
            [
                "trsx_id",
                "cart_id",
                "product_id",
                "amount",
                "price",
                "products_price",
                "products_price_calcule"
            ]
        ]
    )

Tous les montants de ligne respectent : products_price = amount × price.


In [32]:
total_calcule_par_panier = (
    transactions
    .groupby("cart_id", as_index=False)["products_price"]
    .sum()
    .rename(
        columns={
            "products_price": "cart_price_calcule"
        }
    )
)

total_calcule_par_panier["cart_price_calcule"] = (
    total_calcule_par_panier["cart_price_calcule"]
    .round(2)
)

display(total_calcule_par_panier.head())

,cart_id,cart_price_calcule
0,CRT00001,2764.40
1,CRT00002,3063.86
2,CRT00003,3452.26
3,CRT00004,503.52
4,CRT00005,4980.95


In [33]:
nombre_cart_price_par_panier = (
    transactions
    .groupby("cart_id")["cart_price"]
    .nunique()
)

paniers_plusieurs_cart_price = (
    nombre_cart_price_par_panier[
        nombre_cart_price_par_panier > 1
    ]
)

if paniers_plusieurs_cart_price.empty:
    print(
        "Chaque panier possède une seule valeur de cart_price."
    )
else:
    print(
        "Nombre de paniers avec plusieurs valeurs de cart_price :",
        len(paniers_plusieurs_cart_price)
    )

    display(
        transactions[
            transactions["cart_id"].isin(
                paniers_plusieurs_cart_price.index
            )
        ].sort_values("cart_id")
    )

Chaque panier possède une seule valeur de cart_price.


In [34]:
total_declare_par_panier = (
    transactions
    .groupby("cart_id", as_index=False)["cart_price"]
    .first()
)

In [35]:
controle_totaux_paniers = total_declare_par_panier.merge(
    total_calcule_par_panier,
    on="cart_id",
    how="outer"
)

controle_totaux_paniers["ecart"] = (
    controle_totaux_paniers["cart_price"]
    - controle_totaux_paniers["cart_price_calcule"]
).round(2)

masque_total_incoherent = ~np.isclose(
    controle_totaux_paniers["cart_price"],
    controle_totaux_paniers["cart_price_calcule"],
    atol=0.01,
    equal_nan=False
)

paniers_total_incoherent = controle_totaux_paniers[
    masque_total_incoherent
].copy()

if paniers_total_incoherent.empty:
    print(
        "Tous les paniers respectent : "
        "cart_price = somme des products_price."
    )
else:
    print(
        "Nombre de paniers avec un total incohérent :",
        len(paniers_total_incoherent)
    )

    display(paniers_total_incoherent)

Tous les paniers respectent : cart_price = somme des products_price.


In [36]:
controle_prix_catalogue = transactions.merge(
    products_all[
        [
            "product_id",
            "product_price",
            "price_date"
        ]
    ],
    on="product_id",
    how="left",
    validate="many_to_one"
)

controle_prix_catalogue["ecart_prix_catalogue"] = (
    controle_prix_catalogue["price"]
    - controle_prix_catalogue["product_price"]
).round(2)

In [37]:
prix_differents_catalogue = controle_prix_catalogue[
    ~np.isclose(
        controle_prix_catalogue["price"],
        controle_prix_catalogue["product_price"],
        atol=0.01,
        equal_nan=False
    )
].copy()

if prix_differents_catalogue.empty:
    print(
        "Tous les prix transactionnels correspondent "
        "au prix du catalogue."
    )
else:
    print(
        "Nombre de transactions dont le prix diffère "
        "du catalogue :",
        len(prix_differents_catalogue)
    )

    display(
        prix_differents_catalogue[
            [
                "trsx_id",
                "date",
                "product_id",
                "price",
                "product_price",
                "price_date",
                "ecart_prix_catalogue"
            ]
        ].head(20)
    )

Tous les prix transactionnels correspondent au prix du catalogue.


In [38]:
transactions_avant_date_prix = controle_prix_catalogue[
    controle_prix_catalogue["date"]
    < controle_prix_catalogue["price_date"]
].copy()

if transactions_avant_date_prix.empty:
    print(
        "Aucune transaction n'est antérieure "
        "à la date du prix du catalogue."
    )
else:
    print(
        "Nombre de transactions antérieures "
        "à la date du prix catalogue :",
        len(transactions_avant_date_prix)
    )

    display(
        transactions_avant_date_prix[
            [
                "trsx_id",
                "product_id",
                "date",
                "price_date",
                "price",
                "product_price"
            ]
        ].head(20)
    )

Nombre de transactions antérieures à la date du prix catalogue : 9653


,trsx_id,product_id,date,price_date,price,product_price
0,TRX000001,PRD0152,2026-04-12 07:24:58,2026-07-21,336.37,336.37
1,TRX000002,PRD0050,2026-03-31 01:20:13,2026-07-24,164.79,164.79
2,TRX000003,PRD0176,2026-04-29 16:12:53,2026-07-13,288.19,288.19
3,TRX000004,PRD0059,2026-01-14 15:52:09,2026-07-05,252.15,252.15
4,TRX000005,PRD0050,2026-07-07 02:10:57,2026-07-24,164.79,164.79
5,TRX000006,PRD0184,2026-02-18 12:29:39,2026-07-18,85.85,85.85
6,TRX000007,PRD0119,2026-04-03 01:35:54,2026-07-15,332.64,332.64
7,TRX000008,PRD0027,2026-05-30 17:48:12,2026-07-15,209.57,209.57
9,TRX000010,PRD0006,2026-04-27 20:04:57,2026-07-26,207.27,207.27
10,TRX000011,PRD0113,2026-02-01 04:22:05,2026-07-19,310.43,310.43


In [39]:
transactions.drop(
    columns=["products_price_calcule"],
    inplace=True
)

In [40]:
print("\n===== SYNTHÈSE DES RÈGLES MÉTIER =====")

print(
    "Paniers avec plusieurs clients :",
    len(paniers_plusieurs_clients)
)

print(
    "Paniers avec plusieurs dates :",
    len(paniers_plusieurs_dates)
)

print(
    "Quantités invalides :",
    len(transactions_quantite_invalide)
)

print(
    "Prix unitaires invalides :",
    len(transactions_prix_invalide)
)

print(
    "Montants de ligne incohérents :",
    len(transactions_products_price_incoherent)
)

print(
    "Paniers avec plusieurs cart_price :",
    len(paniers_plusieurs_cart_price)
)

print(
    "Totaux de panier incohérents :",
    len(paniers_total_incoherent)
)

print(
    "Prix différents du catalogue :",
    len(prix_differents_catalogue)
)

print(
    "Transactions antérieures à price_date :",
    len(transactions_avant_date_prix)
)


===== SYNTHÈSE DES RÈGLES MÉTIER =====
Paniers avec plusieurs clients : 0
Paniers avec plusieurs dates : 0
Quantités invalides : 0
Prix unitaires invalides : 0
Montants de ligne incohérents : 0
Paniers avec plusieurs cart_price : 0
Totaux de panier incohérents : 0
Prix différents du catalogue : 0
Transactions antérieures à price_date : 9653


In [41]:
table_client = clients[
    [
        "client_id",
        "name",
        "age",
        "sexe",
        "opening_date",
        "status"
    ]
].copy()

In [42]:
print("Dimensions de CLIENT :", table_client.shape)
display(table_client.head())

Dimensions de CLIENT : (435, 6)


,client_id,name,age,sexe,opening_date,status
0,CLI0001,Julie Durand,81,M,2023-03-26,active
1,CLI0002,Laura Bernard,58,M,2022-10-30,active
2,CLI0003,Thomas Moreau,18,M,2025-03-07,active
3,CLI0004,Sophie Moreau,64,M,2025-08-03,active
4,CLI0005,Jean Petit,52,F,2021-06-20,active


In [43]:
table_produit = products_all[
    [
        "product_id",
        "product_name",
        "product_price",
        "price_date"
    ]
].copy()

In [44]:
print("Dimensions de PRODUIT :", table_produit.shape)
display(table_produit.head())

Dimensions de PRODUIT : (200, 4)


,product_id,product_name,product_price,price_date
0,PRD0001,Peinture 125,260.19,2026-07-08
1,PRD0002,Marteau 854,38.12,2026-07-24
2,PRD0003,Peinture 704,149.12,2026-07-01
3,PRD0004,Vis 338,178.11,2026-07-01
4,PRD0005,Vis 833,228.33,2026-07-18


In [45]:
table_panier = (
    transactions[
        [
            "cart_id",
            "date",
            "cart_price",
            "client_id"
        ]
    ]
    .drop_duplicates(subset=["cart_id"])
    .reset_index(drop=True)
)

In [46]:
print("Dimensions de PANIER :", table_panier.shape)
display(table_panier.head())

Dimensions de PANIER : (3333, 4)


,cart_id,date,cart_price,client_id
0,CRT00519,2026-04-12 07:24:58,4507.13,CLI0349
1,CRT00992,2026-03-31 01:20:13,1606.03,CLI0335
2,CRT00421,2026-04-29 16:12:53,6573.80,CLI0246
3,CRT01364,2026-01-14 15:52:09,5839.25,CLI0249
4,CRT02032,2026-07-07 02:10:57,988.74,CLI0057


In [47]:
print(
    "Nombre de paniers :",
    len(table_panier)
)

print(
    "Nombre de cart_id uniques :",
    table_panier["cart_id"].nunique()
)

Nombre de paniers : 3333
Nombre de cart_id uniques : 3333


In [48]:
assert table_panier["cart_id"].is_unique, (
    "Erreur : cart_id n'est pas unique dans la table PANIER."
)

In [49]:
table_ligne_panier = transactions[
    [
        "trsx_id",
        "amount",
        "price",
        "products_price",
        "cart_id",
        "product_id"
    ]
].copy()

In [50]:
print(
    "Dimensions de LIGNE_PANIER :",
    table_ligne_panier.shape
)

display(table_ligne_panier.head())

Dimensions de LIGNE_PANIER : (10343, 6)


,trsx_id,amount,price,products_price,cart_id,product_id
0,TRX000001,10,336.37,3363.70,CRT00519,PRD0152
1,TRX000002,5,164.79,823.95,CRT00992,PRD0050
2,TRX000003,8,288.19,2305.52,CRT00421,PRD0176
3,TRX000004,2,252.15,504.30,CRT01364,PRD0059
4,TRX000005,6,164.79,988.74,CRT02032,PRD0050


In [51]:
cles_primaires = {
    "CLIENT": (
        table_client,
        "client_id"
    ),
    "PRODUIT": (
        table_produit,
        "product_id"
    ),
    "PANIER": (
        table_panier,
        "cart_id"
    ),
    "LIGNE_PANIER": (
        table_ligne_panier,
        "trsx_id"
    )
}

for nom_table, (dataframe, cle) in cles_primaires.items():

    nombre_manquants = dataframe[cle].isna().sum()
    nombre_doublons = dataframe[cle].duplicated().sum()

    if nombre_manquants == 0 and nombre_doublons == 0:
        print(
            f"{nom_table}.{cle} : clé primaire valide."
        )
    else:
        print(
            f"{nom_table}.{cle} : clé primaire invalide."
        )

        print(
            "Valeurs manquantes :",
            nombre_manquants
        )

        print(
            "Valeurs dupliquées :",
            nombre_doublons
        )

CLIENT.client_id : clé primaire valide.
PRODUIT.product_id : clé primaire valide.
PANIER.cart_id : clé primaire valide.
LIGNE_PANIER.trsx_id : clé primaire valide.


In [52]:
paniers_clients_inconnus = table_panier[
    ~table_panier["client_id"].isin(
        table_client["client_id"]
    )
]

print(
    "Paniers avec client inconnu :",
    len(paniers_clients_inconnus)
)

Paniers avec client inconnu : 0


In [53]:
lignes_paniers_inconnus = table_ligne_panier[
    ~table_ligne_panier["cart_id"].isin(
        table_panier["cart_id"]
    )
]

print(
    "Lignes avec panier inconnu :",
    len(lignes_paniers_inconnus)
)

Lignes avec panier inconnu : 0


In [54]:
lignes_produits_inconnus = table_ligne_panier[
    ~table_ligne_panier["product_id"].isin(
        table_produit["product_id"]
    )
]

print(
    "Lignes avec produit inconnu :",
    len(lignes_produits_inconnus)
)

Lignes avec produit inconnu : 0


In [55]:
assert paniers_clients_inconnus.empty, (
    "Erreur d'intégrité : certains paniers référencent "
    "des clients inconnus."
)

assert lignes_paniers_inconnus.empty, (
    "Erreur d'intégrité : certaines lignes référencent "
    "des paniers inconnus."
)

assert lignes_produits_inconnus.empty, (
    "Erreur d'intégrité : certaines lignes référencent "
    "des produits inconnus."
)

In [56]:
dimensions_tables = pd.DataFrame(
    {
        "table": [
            "CLIENT",
            "PRODUIT",
            "PANIER",
            "LIGNE_PANIER"
        ],
        "nombre_lignes": [
            len(table_client),
            len(table_produit),
            len(table_panier),
            len(table_ligne_panier)
        ],
        "nombre_colonnes": [
            table_client.shape[1],
            table_produit.shape[1],
            table_panier.shape[1],
            table_ligne_panier.shape[1]
        ]
    }
)

display(dimensions_tables)

,table,nombre_lignes,nombre_colonnes
0,CLIENT,435,6
1,PRODUIT,200,4
2,PANIER,3333,4
3,LIGNE_PANIER,10343,6


In [57]:
print("===== TABLE CLIENT =====")
display(table_client.head())

print("===== TABLE PRODUIT =====")
display(table_produit.head())

print("===== TABLE PANIER =====")
display(table_panier.head())

print("===== TABLE LIGNE_PANIER =====")
display(table_ligne_panier.head())

===== TABLE CLIENT =====


,client_id,name,age,sexe,opening_date,status
0,CLI0001,Julie Durand,81,M,2023-03-26,active
1,CLI0002,Laura Bernard,58,M,2022-10-30,active
2,CLI0003,Thomas Moreau,18,M,2025-03-07,active
3,CLI0004,Sophie Moreau,64,M,2025-08-03,active
4,CLI0005,Jean Petit,52,F,2021-06-20,active


===== TABLE PRODUIT =====


,product_id,product_name,product_price,price_date
0,PRD0001,Peinture 125,260.19,2026-07-08
1,PRD0002,Marteau 854,38.12,2026-07-24
2,PRD0003,Peinture 704,149.12,2026-07-01
3,PRD0004,Vis 338,178.11,2026-07-01
4,PRD0005,Vis 833,228.33,2026-07-18


===== TABLE PANIER =====


,cart_id,date,cart_price,client_id
0,CRT00519,2026-04-12 07:24:58,4507.13,CLI0349
1,CRT00992,2026-03-31 01:20:13,1606.03,CLI0335
2,CRT00421,2026-04-29 16:12:53,6573.80,CLI0246
3,CRT01364,2026-01-14 15:52:09,5839.25,CLI0249
4,CRT02032,2026-07-07 02:10:57,988.74,CLI0057


===== TABLE LIGNE_PANIER =====


,trsx_id,amount,price,products_price,cart_id,product_id
0,TRX000001,10,336.37,3363.70,CRT00519,PRD0152
1,TRX000002,5,164.79,823.95,CRT00992,PRD0050
2,TRX000003,8,288.19,2305.52,CRT00421,PRD0176
3,TRX000004,2,252.15,504.30,CRT01364,PRD0059
4,TRX000005,6,164.79,988.74,CRT02032,PRD0050


In [58]:
# Chemins des fichiers de sortie
CLIENT_OUTPUT = PROCESSED_DIR / "client.csv"
PRODUIT_OUTPUT = PROCESSED_DIR / "produit.csv"
PANIER_OUTPUT = PROCESSED_DIR / "panier.csv"
LIGNE_PANIER_OUTPUT = PROCESSED_DIR / "ligne_panier.csv"

In [59]:
table_client.to_csv(
    CLIENT_OUTPUT,
    index=False,
    encoding="utf-8-sig"
)

table_produit.to_csv(
    PRODUIT_OUTPUT,
    index=False,
    encoding="utf-8-sig"
)

table_panier.to_csv(
    PANIER_OUTPUT,
    index=False,
    encoding="utf-8-sig"
)

table_ligne_panier.to_csv(
    LIGNE_PANIER_OUTPUT,
    index=False,
    encoding="utf-8-sig"
)

print("Les quatre tables ont été exportées avec succès.")

Les quatre tables ont été exportées avec succès.


In [60]:
fichiers_exportes = [
    CLIENT_OUTPUT,
    PRODUIT_OUTPUT,
    PANIER_OUTPUT,
    LIGNE_PANIER_OUTPUT
]

for fichier in fichiers_exportes:
    print(
        fichier.name,
        "→",
        "OK" if fichier.exists() else "ABSENT"
    )

client.csv → OK
produit.csv → OK
panier.csv → OK
ligne_panier.csv → OK


In [61]:
client_exporte = pd.read_csv(CLIENT_OUTPUT)
produit_exporte = pd.read_csv(PRODUIT_OUTPUT)
panier_exporte = pd.read_csv(PANIER_OUTPUT)
ligne_panier_exporte = pd.read_csv(LIGNE_PANIER_OUTPUT)

print("CLIENT :", client_exporte.shape)
print("PRODUIT :", produit_exporte.shape)
print("PANIER :", panier_exporte.shape)
print("LIGNE_PANIER :", ligne_panier_exporte.shape)

CLIENT : (435, 6)
PRODUIT : (200, 4)
PANIER : (3333, 4)
LIGNE_PANIER : (10343, 6)


In [62]:
assert client_exporte.shape == table_client.shape
assert produit_exporte.shape == table_produit.shape
assert panier_exporte.shape == table_panier.shape
assert ligne_panier_exporte.shape == table_ligne_panier.shape

print("Vérification des exports réussie.")

Vérification des exports réussie.


In [78]:
def creer_ligne_rapport(
    controle,
    categorie,
    nombre_anomalies,
    criticite,
    commentaire
):
    """
    Crée une ligne du rapport qualité en distinguant
    les anomalies bloquantes des avertissements.
    """

    nombre_anomalies = int(nombre_anomalies)

    if nombre_anomalies == 0:
        statut = "CONFORME"

    elif criticite == "Bloquante":
        statut = "ANOMALIE"

    elif criticite == "Avertissement":
        statut = "AVERTISSEMENT"

    else:
        statut = "INFORMATION"

    return {
        "controle": controle,
        "categorie": categorie,
        "nombre_anomalies": nombre_anomalies,
        "statut": statut,
        "criticite": criticite,
        "commentaire": commentaire
    }

In [79]:
rapport_qualite = []

In [80]:
rapport_qualite.append(
    creer_ligne_rapport(
        controle="Valeurs manquantes dans CLIENT",
        categorie="Complétude",
        nombre_anomalies=int(table_client.isna().sum().sum()),
        criticite="Bloquante",
        commentaire="Aucune valeur obligatoire ne doit être absente."
    )
)

rapport_qualite.append(
    creer_ligne_rapport(
        controle="Valeurs manquantes dans PRODUIT",
        categorie="Complétude",
        nombre_anomalies=int(table_produit.isna().sum().sum()),
        criticite="Bloquante",
        commentaire="Aucune valeur obligatoire ne doit être absente."
    )
)

rapport_qualite.append(
    creer_ligne_rapport(
        controle="Valeurs manquantes dans PANIER",
        categorie="Complétude",
        nombre_anomalies=int(table_panier.isna().sum().sum()),
        criticite="Bloquante",
        commentaire="Aucune valeur obligatoire ne doit être absente."
    )
)

rapport_qualite.append(
    creer_ligne_rapport(
        controle="Valeurs manquantes dans LIGNE_PANIER",
        categorie="Complétude",
        nombre_anomalies=int(
            table_ligne_panier.isna().sum().sum()
        ),
        criticite="Bloquante",
        commentaire="Aucune valeur obligatoire ne doit être absente."
    )
)

In [81]:
rapport_qualite.append(
    creer_ligne_rapport(
        controle="Doublons complets dans CLIENT",
        categorie="Unicité",
        nombre_anomalies=int(
            table_client.duplicated().sum()
        ),
        criticite="Bloquante",
        commentaire="Une ligne CLIENT ne doit pas être répétée."
    )
)

rapport_qualite.append(
    creer_ligne_rapport(
        controle="Doublons complets dans PRODUIT",
        categorie="Unicité",
        nombre_anomalies=int(
            table_produit.duplicated().sum()
        ),
        criticite="Bloquante",
        commentaire="Une ligne PRODUIT ne doit pas être répétée."
    )
)

rapport_qualite.append(
    creer_ligne_rapport(
        controle="Doublons complets dans PANIER",
        categorie="Unicité",
        nombre_anomalies=int(
            table_panier.duplicated().sum()
        ),
        criticite="Bloquante",
        commentaire="Une ligne PANIER ne doit pas être répétée."
    )
)

rapport_qualite.append(
    creer_ligne_rapport(
        controle="Doublons complets dans LIGNE_PANIER",
        categorie="Unicité",
        nombre_anomalies=int(
            table_ligne_panier.duplicated().sum()
        ),
        criticite="Bloquante",
        commentaire="Une ligne transactionnelle ne doit pas être répétée."
    )
)

In [82]:
rapport_qualite.append(
    creer_ligne_rapport(
        controle="Unicité de client_id",
        categorie="Clé primaire",
        nombre_anomalies=int(
            table_client["client_id"].duplicated().sum()
            + table_client["client_id"].isna().sum()
        ),
        criticite="Bloquante",
        commentaire="client_id doit être unique et renseigné."
    )
)

rapport_qualite.append(
    creer_ligne_rapport(
        controle="Unicité de product_id",
        categorie="Clé primaire",
        nombre_anomalies=int(
            table_produit["product_id"].duplicated().sum()
            + table_produit["product_id"].isna().sum()
        ),
        criticite="Bloquante",
        commentaire="product_id doit être unique et renseigné."
    )
)

rapport_qualite.append(
    creer_ligne_rapport(
        controle="Unicité de cart_id",
        categorie="Clé primaire",
        nombre_anomalies=int(
            table_panier["cart_id"].duplicated().sum()
            + table_panier["cart_id"].isna().sum()
        ),
        criticite="Bloquante",
        commentaire="cart_id doit être unique et renseigné."
    )
)

rapport_qualite.append(
    creer_ligne_rapport(
        controle="Unicité de trsx_id",
        categorie="Clé primaire",
        nombre_anomalies=int(
            table_ligne_panier["trsx_id"].duplicated().sum()
            + table_ligne_panier["trsx_id"].isna().sum()
        ),
        criticite="Bloquante",
        commentaire="trsx_id doit être unique et renseigné."
    )
)

In [83]:
rapport_qualite.append(
    creer_ligne_rapport(
        controle="PANIER vers CLIENT",
        categorie="Intégrité référentielle",
        nombre_anomalies=len(paniers_clients_inconnus),
        criticite="Bloquante",
        commentaire="Chaque panier doit référencer un client existant."
    )
)

rapport_qualite.append(
    creer_ligne_rapport(
        controle="LIGNE_PANIER vers PANIER",
        categorie="Intégrité référentielle",
        nombre_anomalies=len(lignes_paniers_inconnus),
        criticite="Bloquante",
        commentaire="Chaque ligne doit référencer un panier existant."
    )
)

rapport_qualite.append(
    creer_ligne_rapport(
        controle="LIGNE_PANIER vers PRODUIT",
        categorie="Intégrité référentielle",
        nombre_anomalies=len(lignes_produits_inconnus),
        criticite="Bloquante",
        commentaire="Chaque ligne doit référencer un produit existant."
    )
)

In [84]:
rapport_qualite.append(
    creer_ligne_rapport(
        controle="Un seul client par panier",
        categorie="Cohérence métier",
        nombre_anomalies=len(paniers_plusieurs_clients),
        criticite="Bloquante",
        commentaire="Un panier ne peut appartenir qu'à un seul client."
    )
)

rapport_qualite.append(
    creer_ligne_rapport(
        controle="Une seule date par panier",
        categorie="Cohérence métier",
        nombre_anomalies=len(paniers_plusieurs_dates),
        criticite="Bloquante",
        commentaire="Un panier ne peut avoir qu'une seule date."
    )
)

rapport_qualite.append(
    creer_ligne_rapport(
        controle="Quantités strictement positives",
        categorie="Validité métier",
        nombre_anomalies=len(
            transactions_quantite_invalide
        ),
        criticite="Bloquante",
        commentaire="La quantité doit être strictement supérieure à zéro."
    )
)

rapport_qualite.append(
    creer_ligne_rapport(
        controle="Prix unitaires valides",
        categorie="Validité métier",
        nombre_anomalies=len(
            transactions_prix_invalide
        ),
        criticite="Bloquante",
        commentaire="Le prix unitaire doit être supérieur ou égal à zéro."
    )
)

rapport_qualite.append(
    creer_ligne_rapport(
        controle="Calcul de products_price",
        categorie="Exactitude",
        nombre_anomalies=len(
            transactions_products_price_incoherent
        ),
        criticite="Bloquante",
        commentaire="products_price doit être égal à amount multiplié par price."
    )
)

rapport_qualite.append(
    creer_ligne_rapport(
        controle="Un seul cart_price par panier",
        categorie="Cohérence métier",
        nombre_anomalies=len(
            paniers_plusieurs_cart_price
        ),
        criticite="Bloquante",
        commentaire="Toutes les lignes d'un panier doivent partager le même total."
    )
)

rapport_qualite.append(
    creer_ligne_rapport(
        controle="Calcul du total du panier",
        categorie="Exactitude",
        nombre_anomalies=len(
            paniers_total_incoherent
        ),
        criticite="Bloquante",
        commentaire="cart_price doit être égal à la somme des products_price."
    )
)

In [85]:
rapport_qualite.append(
    creer_ligne_rapport(
        controle="Écart entre prix transactionnel et catalogue",
        categorie="Cohérence tarifaire",
        nombre_anomalies=len(
            prix_differents_catalogue
        ),
        criticite="Avertissement",
        commentaire=(
            "Un écart peut être lié à une remise ou à un prix historique. "
            "Le prix transactionnel est conservé."
        )
    )
)

rapport_qualite.append(
    creer_ligne_rapport(
        controle="Transaction antérieure à price_date",
        categorie="Cohérence temporelle",
        nombre_anomalies=len(
            transactions_avant_date_prix
        ),
        criticite="Avertissement",
        commentaire=(
            "Ce contrôle constitue un signalement et non une erreur bloquante."
        )
    )
)

rapport_qualite.append(
    creer_ligne_rapport(
        controle="Répétition du couple cart_id-product_id",
        categorie="Information",
        nombre_anomalies=0,
        criticite="Information",
        commentaire=(
            f"{nombre_couples_repetes} couples répétés et "
            f"{nombre_lignes_repetees} lignes concernées. "
            "Ces lignes possèdent des trsx_id distincts et ne sont pas "
            "considérées comme des doublons."
        )
    )
)

In [86]:
rapport_qualite_df = pd.DataFrame(rapport_qualite)

display(rapport_qualite_df)

,controle,categorie,nombre_anomalies,statut,criticite,commentaire
0,Valeurs manquantes dans CLIENT,Complétude,0,CONFORME,Bloquante,Aucune valeur obligatoire ne doit être absente.
1,Valeurs manquantes dans PRODUIT,Complétude,0,CONFORME,Bloquante,Aucune valeur obligatoire ne doit être absente.
2,Valeurs manquantes dans PANIER,Complétude,0,CONFORME,Bloquante,Aucune valeur obligatoire ne doit être absente.
3,Valeurs manquantes dans LIGNE_PANIER,Complétude,0,CONFORME,Bloquante,Aucune valeur obligatoire ne doit être absente.
4,Doublons complets dans CLIENT,Unicité,0,CONFORME,Bloquante,Une ligne CLIENT ne doit pas être répétée.
5,Doublons complets dans PRODUIT,Unicité,0,CONFORME,Bloquante,Une ligne PRODUIT ne doit pas être répétée.
6,Doublons complets dans PANIER,Unicité,0,CONFORME,Bloquante,Une ligne PANIER ne doit pas être répétée.
7,Doublons complets dans LIGNE_PANIER,Unicité,0,CONFORME,Bloquante,Une ligne transactionnelle ne doit pas être ré...
8,Unicité de client_id,Clé primaire,0,CONFORME,Bloquante,client_id doit être unique et renseigné.
9,Unicité de product_id,Clé primaire,0,CONFORME,Bloquante,product_id doit être unique et renseigné.


In [87]:
date_execution = pd.Timestamp.now().strftime(
    "%Y-%m-%d %H:%M:%S"
)

rapport_qualite_df.insert(
    0,
    "date_execution",
    date_execution
)

In [88]:
RAPPORT_QUALITE_OUTPUT = (
    REPORTS_DIR / "rapport_qualite.csv"
)

rapport_qualite_df.to_csv(
    RAPPORT_QUALITE_OUTPUT,
    index=False,
    encoding="utf-8-sig"
)

print(
    "Rapport qualité exporté :",
    RAPPORT_QUALITE_OUTPUT
)

Rapport qualité exporté : C:\Users\abide\projet_lerouge_moulin\reports\rapport_qualite.csv


In [89]:
synthese_statuts = (
    rapport_qualite_df["statut"]
    .value_counts()
    .rename_axis("statut")
    .reset_index(name="nombre_controles")
)

display(synthese_statuts)

,statut,nombre_controles
0,CONFORME,24
1,AVERTISSEMENT,1


In [90]:
nombre_controles = len(rapport_qualite_df)

nombre_controles_conformes = (
    rapport_qualite_df["statut"]
    .eq("CONFORME")
    .sum()
)

nombre_controles_anomalies = (
    rapport_qualite_df["statut"]
    .eq("ANOMALIE")
    .sum()
)

print("Nombre total de contrôles :", nombre_controles)
print(
    "Nombre de contrôles conformes :",
    nombre_controles_conformes
)
print(
    "Nombre de contrôles avec anomalie :",
    nombre_controles_anomalies
)

Nombre total de contrôles : 25
Nombre de contrôles conformes : 24
Nombre de contrôles avec anomalie : 0


In [91]:
anomalies_bloquantes = rapport_qualite_df[
    (rapport_qualite_df["criticite"] == "Bloquante")
    & (rapport_qualite_df["nombre_anomalies"] > 0)
]

if anomalies_bloquantes.empty:
    statut_pipeline = "SUCCÈS"

    print(
        "Pipeline terminé avec succès : "
        "aucune anomalie bloquante détectée."
    )
else:
    statut_pipeline = "ÉCHEC"

    print(
        "Pipeline interrompu : "
        "des anomalies bloquantes ont été détectées."
    )

    display(anomalies_bloquantes)

Pipeline terminé avec succès : aucune anomalie bloquante détectée.


In [92]:
rapport_qualite_df[
    rapport_qualite_df["statut"] == "ANOMALIE"
][
    [
        "controle",
        "categorie",
        "nombre_anomalies",
        "criticite",
        "commentaire"
    ]
]

,controle,categorie,nombre_anomalies,criticite,commentaire


# Version automatisée et réutilisable du pipeline

In [93]:
from pathlib import Path
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd

In [94]:
PROJECT_DIR = Path.cwd().parent

RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
REPORTS_DIR = PROJECT_DIR / "reports"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

In [95]:
CLIENT_FILE = RAW_DIR / "client.xml"
PRODUCT_CSV_FILE = RAW_DIR / "product.csv"
PRODUCT_XML_FILE = RAW_DIR / "product.xml"
TRANSACTIONS_FILE = RAW_DIR / "transactions_2026.csv"

In [96]:
def lire_xml(chemin_fichier, balise_element):
    """
    Lit les enregistrements d'un fichier XML
    et les transforme en DataFrame pandas.
    """

    arbre = ET.parse(chemin_fichier)
    racine = arbre.getroot()

    donnees = []

    for element in racine.findall(balise_element):
        ligne = {
            champ.tag: champ.text
            for champ in element
        }

        donnees.append(ligne)

    return pd.DataFrame(donnees)

In [97]:
d_cdef charger_donnees(raw_dir):
    """
    Charge les quatre fichiers sources du projet.

    Retourne
    --------
    clients : DataFrame
    products_csv : DataFrame
    products_xml : DataFrame
    transactions : DataFrame
    """

    fichiers = {
        "client_xml": raw_dir / "client.xml",
        "product_csv": raw_dir / "product.csv",
        "product_xml": raw_dir / "product.xml",
        "transactions_csv": raw_dir / "transactions_2026.csv"
    }

    fichiers_absents = [
        str(chemin)
        for chemin in fichiers.values()
        if not chemin.exists()
    ]

    if fichiers_absents:
        raise FileNotFoundError(
            "Fichiers sources absents :\n"
            + "\n".join(fichiers_absents)
        )

    clients = lire_xml(
        fichiers["client_xml"],
        "Client"
    )

    products_csv = pd.reasv(
        fichiers["product_csv"]
    )

    products_xml = lire_xml(
        fichiers["product_xml"],
        "Product"
    )

    transactions = pd.read_csv(
        fichiers["transactions_csv"]
    )

    print("Chargement des données terminé.")

    print(f"CLIENT XML : {len(clients)} lignes")
    print(f"PRODUIT CSV : {len(products_csv)} lignes")
    print(f"PRODUIT XML : {len(products_xml)} lignes")
    print(f"TRANSACTIONS : {len(transactions)} lignes")

    return (
        clients,
        products_csv,
        products_xml,
        transactions
    )

In [98]:
(
    clients_test,
    products_csv_test,
    products_xml_test,
    transactions_test
) = charger_donnees(RAW_DIR)

Chargement des données terminé.
CLIENT XML : 435 lignes
PRODUIT CSV : 180 lignes
PRODUIT XML : 20 lignes
TRANSACTIONS : 10343 lignes


In [99]:
def convertir_types(
    clients,
    products_csv,
    products_xml,
    transactions
):
    """
    Convertit et harmonise les types des données sources.
    Concatène également les deux sources PRODUIT.
    """

    clients = clients.copy()
    products_csv = products_csv.copy()
    products_xml = products_xml.copy()
    transactions = transactions.copy()

    # CLIENT
    clients["client_id"] = clients["client_id"].astype("string")
    clients["name"] = clients["name"].astype("string")
    clients["sexe"] = clients["sexe"].astype("string")
    clients["status"] = clients["status"].astype("string")

    clients["age"] = pd.to_numeric(
        clients["age"],
        errors="coerce"
    ).astype("Int64")

    clients["opening_date"] = pd.to_datetime(
        clients["opening_date"],
        errors="coerce"
    )

    # PRODUIT CSV et XML
    for dataframe in [products_csv, products_xml]:

        dataframe["product_id"] = (
            dataframe["product_id"].astype("string")
        )

        dataframe["product_name"] = (
            dataframe["product_name"].astype("string")
        )

        dataframe["product_price"] = pd.to_numeric(
            dataframe["product_price"],
            errors="coerce"
        )

        dataframe["price_date"] = pd.to_datetime(
            dataframe["price_date"],
            errors="coerce"
        )

    products_all = pd.concat(
        [products_csv, products_xml],
        ignore_index=True
    )

    # TRANSACTIONS
    colonnes_texte = [
        "trsx_id",
        "client_id",
        "cart_id",
        "product_id"
    ]

    for colonne in colonnes_texte:
        transactions[colonne] = (
            transactions[colonne].astype("string")
        )

    transactions["date"] = pd.to_datetime(
        transactions["date"],
        errors="coerce"
    )

    transactions["amount"] = pd.to_numeric(
        transactions["amount"],
        errors="coerce"
    ).astype("Int64")

    colonnes_numeriques = [
        "price",
        "products_price",
        "cart_price"
    ]

    for colonne in colonnes_numeriques:
        transactions[colonne] = pd.to_numeric(
            transactions[colonne],
            errors="coerce"
        )

    print("Conversion des types terminée.")

    return clients, products_all, transactions

In [100]:
clients_test, products_test, transactions_test = convertir_types(
    clients_test,
    products_csv_test,
    products_xml_test,
    transactions_test
)

Conversion des types terminée.


In [101]:
print(clients_test.dtypes)
print()
print(products_test.dtypes)
print()
print(transactions_test.dtypes)

client_id       string[python]
name            string[python]
age                      Int64
sexe            string[python]
opening_date    datetime64[ns]
status          string[python]
dtype: object

product_id       string[python]
product_name     string[python]
product_price           float64
price_date       datetime64[ns]
dtype: object

trsx_id           string[python]
client_id         string[python]
cart_id           string[python]
product_id        string[python]
date              datetime64[ns]
amount                     Int64
price                    float64
products_price           float64
cart_price               float64
dtype: object


In [102]:
def transformer_donnees(
    clients,
    products_all,
    transactions
):
    """
    Construit les tables relationnelles finales :
    CLIENT, PRODUIT, PANIER et LIGNE_PANIER.
    """

    table_client = clients[
        [
            "client_id",
            "name",
            "age",
            "sexe",
            "opening_date",
            "status"
        ]
    ].copy()

    table_produit = products_all[
        [
            "product_id",
            "product_name",
            "product_price",
            "price_date"
        ]
    ].copy()

    table_panier = (
        transactions[
            [
                "cart_id",
                "date",
                "cart_price",
                "client_id"
            ]
        ]
        .drop_duplicates(subset=["cart_id"])
        .reset_index(drop=True)
    )

    table_ligne_panier = transactions[
        [
            "trsx_id",
            "amount",
            "price",
            "products_price",
            "cart_id",
            "product_id"
        ]
    ].copy()

    print("Transformation des données terminée.")

    return {
        "CLIENT": table_client,
        "PRODUIT": table_produit,
        "PANIER": table_panier,
        "LIGNE_PANIER": table_ligne_panier
    }

In [103]:
tables_test = transformer_donnees(
    clients_test,
    products_test,
    transactions_test
)

Transformation des données terminée.


In [104]:
for nom_table, dataframe in tables_test.items():
    print(
        nom_table,
        "→",
        dataframe.shape
    )

CLIENT → (435, 6)
PRODUIT → (200, 4)
PANIER → (3333, 4)
LIGNE_PANIER → (10343, 6)


In [105]:
def verifier_qualite(
    clients,
    products_all,
    transactions
):
    """
    Exécute tous les contrôles qualité du pipeline.

    Paramètres
    ----------
    clients : DataFrame
    products_all : DataFrame
    transactions : DataFrame

    Retour
    ------
    rapport_qualite_df : DataFrame
    statut_pipeline : str
    """

    rapport_qualite = []

    # Les contrôles seront ajoutés ici

    rapport_qualite_df = pd.DataFrame(rapport_qualite)

    anomalies_bloquantes = rapport_qualite_df[
        (rapport_qualite_df["criticite"] == "Bloquante")
        & (rapport_qualite_df["nombre_anomalies"] > 0)
    ]

    if anomalies_bloquantes.empty:
        statut_pipeline = "SUCCÈS"
    else:
        statut_pipeline = "ÉCHEC"

    return rapport_qualite_df, statut_pipeline

## Fonctions réutilisables de contrôle qualité

In [106]:
def creer_ligne_rapport(
    controle,
    categorie,
    nombre_anomalies,
    criticite,
    commentaire
):
    """
    Crée une ligne standardisée pour le rapport qualité.
    """

    nombre_anomalies = int(nombre_anomalies)

    if nombre_anomalies == 0:
        statut = "CONFORME"
    elif criticite == "Bloquante":
        statut = "ANOMALIE"
    elif criticite == "Avertissement":
        statut = "AVERTISSEMENT"
    else:
        statut = "INFORMATION"

    return {
        "controle": controle,
        "categorie": categorie,
        "nombre_anomalies": nombre_anomalies,
        "statut": statut,
        "criticite": criticite,
        "commentaire": commentaire
    }

In [107]:
def compter_valeurs_manquantes(dataframe):
    """
    Retourne le nombre total de valeurs manquantes.
    """
    return int(dataframe.isna().sum().sum())

In [108]:
def compter_doublons_complets(dataframe):
    """
    Retourne le nombre de lignes entièrement dupliquées.
    """
    return int(dataframe.duplicated().sum())

In [109]:
def compter_anomalies_cle_primaire(dataframe, colonne_cle):
    """
    Compte les clés primaires dupliquées ou manquantes.
    """
    doublons = dataframe[colonne_cle].duplicated().sum()
    valeurs_manquantes = dataframe[colonne_cle].isna().sum()

    return int(doublons + valeurs_manquantes)

In [110]:
def compter_cles_etrangeres_inconnues(
    dataframe_enfant,
    colonne_etrangere,
    dataframe_parent,
    colonne_parent
):
    """
    Compte les clés étrangères qui ne correspondent
    à aucune clé de la table parente.
    """

    masque_inconnu = ~dataframe_enfant[colonne_etrangere].isin(
        dataframe_parent[colonne_parent]
    )

    return int(masque_inconnu.sum())

In [111]:
def compter_paniers_plusieurs_clients(transactions):
    """
    Compte les paniers associés à plusieurs clients.
    """

    nombre_clients_par_panier = (
        transactions
        .groupby("cart_id")["client_id"]
        .nunique()
    )

    return int((nombre_clients_par_panier > 1).sum())

In [112]:
def compter_paniers_plusieurs_dates(transactions):
    """
    Compte les paniers associés à plusieurs dates.
    """

    nombre_dates_par_panier = (
        transactions
        .groupby("cart_id")["date"]
        .nunique()
    )

    return int((nombre_dates_par_panier > 1).sum())

In [113]:
def compter_quantites_invalides(transactions):
    """
    Compte les quantités nulles, négatives ou manquantes.
    """

    masque_invalide = (
        transactions["amount"].isna()
        | (transactions["amount"] <= 0)
    )

    return int(masque_invalide.sum())

In [114]:
def compter_prix_invalides(transactions):
    """
    Compte les prix unitaires négatifs ou manquants.
    """

    masque_invalide = (
        transactions["price"].isna()
        | (transactions["price"] < 0)
    )

    return int(masque_invalide.sum())

In [115]:
def compter_products_price_incoherents(transactions):
    """
    Vérifie que products_price = amount × price.
    """

    montant_calcule = (
        transactions["amount"].astype("float64")
        * transactions["price"]
    )

    masque_incoherent = ~np.isclose(
        transactions["products_price"],
        montant_calcule,
        equal_nan=False
    )

    return int(masque_incoherent.sum())

In [116]:
def compter_paniers_plusieurs_totaux(transactions):
    """
    Compte les paniers ayant plusieurs valeurs de cart_price.
    """

    nombre_totaux_par_panier = (
        transactions
        .groupby("cart_id")["cart_price"]
        .nunique()
    )

    return int((nombre_totaux_par_panier > 1).sum())

In [117]:
def compter_totaux_panier_incoherents(transactions):
    """
    Vérifie que cart_price correspond à la somme
    des products_price du panier.
    """

    totaux_calcules = (
        transactions
        .groupby("cart_id", as_index=False)["products_price"]
        .sum()
        .rename(columns={
            "products_price": "cart_price_calcule"
        })
    )

    totaux_declares = (
        transactions[
            ["cart_id", "cart_price"]
        ]
        .drop_duplicates(subset=["cart_id"])
    )

    comparaison = totaux_declares.merge(
        totaux_calcules,
        on="cart_id",
        how="left"
    )

    masque_incoherent = ~np.isclose(
        comparaison["cart_price"],
        comparaison["cart_price_calcule"],
        equal_nan=False
    )

    return int(masque_incoherent.sum())

In [118]:
def compter_ecarts_prix_catalogue(
    transactions,
    products_all
):
    """
    Compte les transactions dont le prix diffère
    du prix présent dans le catalogue.
    """

    comparaison = transactions.merge(
        products_all[
            [
                "product_id",
                "product_price"
            ]
        ],
        on="product_id",
        how="left"
    )

    masque_ecart = ~np.isclose(
        comparaison["price"],
        comparaison["product_price"],
        equal_nan=False
    )

    return int(masque_ecart.sum())

In [119]:
def compter_transactions_avant_date_prix(
    transactions,
    products_all
):
    """
    Compte les transactions antérieures à price_date.
    """

    comparaison = transactions.merge(
        products_all[
            [
                "product_id",
                "price_date"
            ]
        ],
        on="product_id",
        how="left"
    )

    masque = (
        comparaison["date"].notna()
        & comparaison["price_date"].notna()
        & (comparaison["date"] < comparaison["price_date"])
    )

    return int(masque.sum())

In [120]:
def analyser_repetitions_cart_produit(transactions):
    """
    Retourne le nombre de couples cart_id-product_id répétés
    et le nombre de lignes concernées.
    """

    masque_repetition = transactions.duplicated(
        subset=["cart_id", "product_id"],
        keep=False
    )

    lignes_repetees = transactions.loc[
        masque_repetition,
        ["cart_id", "product_id"]
    ]

    nombre_lignes = len(lignes_repetees)

    nombre_couples = (
        lignes_repetees
        .drop_duplicates()
        .shape[0]
    )

    return int(nombre_couples), int(nombre_lignes)

In [121]:
def verifier_qualite(
    clients,
    products_all,
    transactions,
    tables
):
    """
    Exécute l'ensemble des contrôles qualité
    et retourne le rapport ainsi que le statut global.
    """

    table_client = tables["CLIENT"]
    table_produit = tables["PRODUIT"]
    table_panier = tables["PANIER"]
    table_ligne_panier = tables["LIGNE_PANIER"]

    rapport = []

    # Complétude
    for nom_table, dataframe in tables.items():

        rapport.append(
            creer_ligne_rapport(
                controle=f"Valeurs manquantes dans {nom_table}",
                categorie="Complétude",
                nombre_anomalies=compter_valeurs_manquantes(
                    dataframe
                ),
                criticite="Bloquante",
                commentaire=(
                    "Les colonnes obligatoires doivent être renseignées."
                )
            )
        )

    # Doublons complets
    for nom_table, dataframe in tables.items():

        rapport.append(
            creer_ligne_rapport(
                controle=f"Doublons complets dans {nom_table}",
                categorie="Unicité",
                nombre_anomalies=compter_doublons_complets(
                    dataframe
                ),
                criticite="Bloquante",
                commentaire=(
                    "Une ligne complète ne doit pas être répétée."
                )
            )
        )

    # Clés primaires
    cles_primaires = {
        "CLIENT": "client_id",
        "PRODUIT": "product_id",
        "PANIER": "cart_id",
        "LIGNE_PANIER": "trsx_id"
    }

    for nom_table, colonne_cle in cles_primaires.items():

        rapport.append(
            creer_ligne_rapport(
                controle=f"Unicité de {colonne_cle}",
                categorie="Clé primaire",
                nombre_anomalies=compter_anomalies_cle_primaire(
                    tables[nom_table],
                    colonne_cle
                ),
                criticite="Bloquante",
                commentaire=(
                    f"{colonne_cle} doit être unique et renseigné."
                )
            )
        )

    # Intégrité référentielle
    rapport.append(
        creer_ligne_rapport(
            controle="PANIER vers CLIENT",
            categorie="Intégrité référentielle",
            nombre_anomalies=compter_cles_etrangeres_inconnues(
                table_panier,
                "client_id",
                table_client,
                "client_id"
            ),
            criticite="Bloquante",
            commentaire=(
                "Chaque panier doit référencer un client existant."
            )
        )
    )

    rapport.append(
        creer_ligne_rapport(
            controle="LIGNE_PANIER vers PANIER",
            categorie="Intégrité référentielle",
            nombre_anomalies=compter_cles_etrangeres_inconnues(
                table_ligne_panier,
                "cart_id",
                table_panier,
                "cart_id"
            ),
            criticite="Bloquante",
            commentaire=(
                "Chaque ligne doit référencer un panier existant."
            )
        )
    )

    rapport.append(
        creer_ligne_rapport(
            controle="LIGNE_PANIER vers PRODUIT",
            categorie="Intégrité référentielle",
            nombre_anomalies=compter_cles_etrangeres_inconnues(
                table_ligne_panier,
                "product_id",
                table_produit,
                "product_id"
            ),
            criticite="Bloquante",
            commentaire=(
                "Chaque ligne doit référencer un produit existant."
            )
        )
    )

    # Règles métier
    controles_metier = [
        (
            "Un seul client par panier",
            "Cohérence métier",
            compter_paniers_plusieurs_clients(transactions),
            "Un panier ne peut appartenir qu'à un seul client."
        ),
        (
            "Une seule date par panier",
            "Cohérence métier",
            compter_paniers_plusieurs_dates(transactions),
            "Un panier ne peut avoir qu'une seule date."
        ),
        (
            "Quantités strictement positives",
            "Validité métier",
            compter_quantites_invalides(transactions),
            "La quantité doit être strictement positive."
        ),
        (
            "Prix unitaires valides",
            "Validité métier",
            compter_prix_invalides(transactions),
            "Le prix doit être positif ou nul."
        ),
        (
            "Calcul de products_price",
            "Exactitude",
            compter_products_price_incoherents(transactions),
            "products_price doit être égal à amount multiplié par price."
        ),
        (
            "Un seul cart_price par panier",
            "Cohérence métier",
            compter_paniers_plusieurs_totaux(transactions),
            "Un panier ne doit posséder qu'un seul total déclaré."
        ),
        (
            "Calcul du total du panier",
            "Exactitude",
            compter_totaux_panier_incoherents(transactions),
            "cart_price doit correspondre à la somme des products_price."
        )
    ]

    for (
        controle,
        categorie,
        nombre_anomalies,
        commentaire
    ) in controles_metier:

        rapport.append(
            creer_ligne_rapport(
                controle=controle,
                categorie=categorie,
                nombre_anomalies=nombre_anomalies,
                criticite="Bloquante",
                commentaire=commentaire
            )
        )

    # Avertissements
    rapport.append(
        creer_ligne_rapport(
            controle="Écart entre prix transactionnel et catalogue",
            categorie="Cohérence tarifaire",
            nombre_anomalies=compter_ecarts_prix_catalogue(
                transactions,
                products_all
            ),
            criticite="Avertissement",
            commentaire=(
                "Un écart peut correspondre à une remise "
                "ou à un prix historique."
            )
        )
    )

    rapport.append(
        creer_ligne_rapport(
            controle="Transaction antérieure à price_date",
            categorie="Cohérence temporelle",
            nombre_anomalies=compter_transactions_avant_date_prix(
                transactions,
                products_all
            ),
            criticite="Avertissement",
            commentaire=(
                "Le catalogue ne contenant pas d'historique tarifaire, "
                "le prix transactionnel est conservé."
            )
        )
    )

    # Information sur les répétitions
    nombre_couples, nombre_lignes = (
        analyser_repetitions_cart_produit(transactions)
    )

    rapport.append(
        creer_ligne_rapport(
            controle="Répétition du couple cart_id-product_id",
            categorie="Information",
            nombre_anomalies=0,
            criticite="Information",
            commentaire=(
                f"{nombre_couples} couples répétés et "
                f"{nombre_lignes} lignes concernées. "
                "Les trsx_id sont distincts : "
                "ces lignes ne sont pas considérées comme des doublons."
            )
        )
    )

    rapport_qualite_df = pd.DataFrame(rapport)

    rapport_qualite_df.insert(
        0,
        "date_execution",
        pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S")
    )

    anomalies_bloquantes = rapport_qualite_df[
        (rapport_qualite_df["criticite"] == "Bloquante")
        & (rapport_qualite_df["nombre_anomalies"] > 0)
    ]

    statut_pipeline = (
        "SUCCÈS"
        if anomalies_bloquantes.empty
        else "ÉCHEC"
    )

    return (
        rapport_qualite_df,
        statut_pipeline,
        anomalies_bloquantes
    )

In [122]:
rapport_test, statut_test, anomalies_test = verifier_qualite(
    clients_test,
    products_test,
    transactions_test,
    tables_test
)

In [123]:
display(rapport_test)

,date_execution,controle,categorie,nombre_anomalies,statut,criticite,commentaire
0,2026-07-28 09:30:25,Valeurs manquantes dans CLIENT,Complétude,0,CONFORME,Bloquante,Les colonnes obligatoires doivent être renseig...
1,2026-07-28 09:30:25,Valeurs manquantes dans PRODUIT,Complétude,0,CONFORME,Bloquante,Les colonnes obligatoires doivent être renseig...
2,2026-07-28 09:30:25,Valeurs manquantes dans PANIER,Complétude,0,CONFORME,Bloquante,Les colonnes obligatoires doivent être renseig...
3,2026-07-28 09:30:25,Valeurs manquantes dans LIGNE_PANIER,Complétude,0,CONFORME,Bloquante,Les colonnes obligatoires doivent être renseig...
4,2026-07-28 09:30:25,Doublons complets dans CLIENT,Unicité,0,CONFORME,Bloquante,Une ligne complète ne doit pas être répétée.
5,2026-07-28 09:30:25,Doublons complets dans PRODUIT,Unicité,0,CONFORME,Bloquante,Une ligne complète ne doit pas être répétée.
6,2026-07-28 09:30:25,Doublons complets dans PANIER,Unicité,0,CONFORME,Bloquante,Une ligne complète ne doit pas être répétée.
7,2026-07-28 09:30:25,Doublons complets dans LIGNE_PANIER,Unicité,0,CONFORME,Bloquante,Une ligne complète ne doit pas être répétée.
8,2026-07-28 09:30:25,Unicité de client_id,Clé primaire,0,CONFORME,Bloquante,client_id doit être unique et renseigné.
9,2026-07-28 09:30:25,Unicité de product_id,Clé primaire,0,CONFORME,Bloquante,product_id doit être unique et renseigné.


In [124]:
print("Statut du pipeline :", statut_test)

Statut du pipeline : SUCCÈS


In [125]:
display(
    rapport_test["statut"]
    .value_counts()
    .rename_axis("statut")
    .reset_index(name="nombre_controles")
)

,statut,nombre_controles
0,CONFORME,24
1,AVERTISSEMENT,1


In [126]:
display(anomalies_test)

,date_execution,controle,categorie,nombre_anomalies,statut,criticite,commentaire


In [127]:
def exporter_tables(tables, processed_dir):
    """
    Exporte les tables finales au format CSV.

    Paramètres
    ----------
    tables : dict
        Dictionnaire contenant les DataFrames CLIENT,
        PRODUIT, PANIER et LIGNE_PANIER.
    processed_dir : Path
        Dossier de destination des fichiers transformés.

    Retour
    ------
    dict
        Chemins des fichiers exportés.
    """

    processed_dir.mkdir(parents=True, exist_ok=True)

    noms_fichiers = {
        "CLIENT": "client.csv",
        "PRODUIT": "produit.csv",
        "PANIER": "panier.csv",
        "LIGNE_PANIER": "ligne_panier.csv"
    }

    fichiers_exportes = {}

    for nom_table, dataframe in tables.items():

        chemin_sortie = (
            processed_dir / noms_fichiers[nom_table]
        )

        dataframe.to_csv(
            chemin_sortie,
            index=False,
            encoding="utf-8-sig"
        )

        fichiers_exportes[nom_table] = chemin_sortie

        print(
            f"{nom_table} exportée : "
            f"{chemin_sortie.name}"
        )

    return fichiers_exportes

In [128]:
fichiers_tables_test = exporter_tables(
    tables_test,
    PROCESSED_DIR
)

CLIENT exportée : client.csv
PRODUIT exportée : produit.csv
PANIER exportée : panier.csv
LIGNE_PANIER exportée : ligne_panier.csv


In [129]:
for nom_table, chemin in fichiers_tables_test.items():
    print(
        nom_table,
        "→",
        "OK" if chemin.exists() else "ABSENT"
    )

CLIENT → OK
PRODUIT → OK
PANIER → OK
LIGNE_PANIER → OK


In [130]:
def generer_rapport(
    rapport_qualite_df,
    reports_dir
):
    """
    Exporte le rapport qualité au format CSV.

    Paramètres
    ----------
    rapport_qualite_df : DataFrame
        Rapport produit par verifier_qualite().
    reports_dir : Path
        Dossier de destination du rapport.

    Retour
    ------
    Path
        Chemin du rapport exporté.
    """

    reports_dir.mkdir(parents=True, exist_ok=True)

    chemin_rapport = (
        reports_dir / "rapport_qualite.csv"
    )

    rapport_qualite_df.to_csv(
        chemin_rapport,
        index=False,
        encoding="utf-8-sig"
    )

    print(
        "Rapport qualité exporté :",
        chemin_rapport.name
    )

    return chemin_rapport

In [131]:
rapport_exporte_test = generer_rapport(
    rapport_test,
    REPORTS_DIR
)

Rapport qualité exporté : rapport_qualite.csv


In [132]:
print(
    "Rapport présent :",
    rapport_exporte_test.exists()
)

Rapport présent : True


In [133]:
def executer_pipeline(
    raw_dir,
    processed_dir,
    reports_dir
):
    """
    Exécute l'intégralité du pipeline de données.

    Étapes :
    1. Chargement des sources
    2. Conversion des types
    3. Transformation des données
    4. Contrôles qualité
    5. Génération du rapport
    6. Export des tables si les contrôles bloquants sont conformes

    Retour
    ------
    dict
        Résultats principaux de l'exécution.
    """

    print("=" * 60)
    print("DÉMARRAGE DU PIPELINE LEROUGE MOULIN")
    print("=" * 60)

    try:
        # 1. Chargement
        (
            clients,
            products_csv,
            products_xml,
            transactions
        ) = charger_donnees(raw_dir)

        # 2. Conversion
        (
            clients,
            products_all,
            transactions
        ) = convertir_types(
            clients,
            products_csv,
            products_xml,
            transactions
        )

        # 3. Transformation
        tables = transformer_donnees(
            clients,
            products_all,
            transactions
        )

        # 4. Contrôles qualité
        (
            rapport_qualite_df,
            statut_pipeline,
            anomalies_bloquantes
        ) = verifier_qualite(
            clients,
            products_all,
            transactions,
            tables
        )

        # 5. Le rapport est toujours exporté
        chemin_rapport = generer_rapport(
            rapport_qualite_df,
            reports_dir
        )

        fichiers_exportes = {}

        # 6. Export conditionnel des tables
        if statut_pipeline == "SUCCÈS":

            fichiers_exportes = exporter_tables(
                tables,
                processed_dir
            )

            print()
            print(
                "Pipeline terminé avec succès : "
                "aucune anomalie bloquante détectée."
            )

        else:

            print()
            print(
                "Pipeline interrompu : "
                "des anomalies bloquantes ont été détectées."
            )

            display(anomalies_bloquantes)

        print("=" * 60)

        return {
            "statut": statut_pipeline,
            "tables": tables,
            "rapport_qualite": rapport_qualite_df,
            "anomalies_bloquantes": anomalies_bloquantes,
            "fichiers_exportes": fichiers_exportes,
            "chemin_rapport": chemin_rapport
        }

    except Exception as erreur:

        print()
        print("ÉCHEC TECHNIQUE DU PIPELINE")
        print(type(erreur).__name__, ":", erreur)
        print("=" * 60)

        raise

In [134]:
resultats_pipeline = executer_pipeline(
    RAW_DIR,
    PROCESSED_DIR,
    REPORTS_DIR
)

DÉMARRAGE DU PIPELINE LEROUGE MOULIN
Chargement des données terminé.
CLIENT XML : 435 lignes
PRODUIT CSV : 180 lignes
PRODUIT XML : 20 lignes
TRANSACTIONS : 10343 lignes
Conversion des types terminée.
Transformation des données terminée.
Rapport qualité exporté : rapport_qualite.csv
CLIENT exportée : client.csv
PRODUIT exportée : produit.csv
PANIER exportée : panier.csv
LIGNE_PANIER exportée : ligne_panier.csv

Pipeline terminé avec succès : aucune anomalie bloquante détectée.


In [135]:
print(
    "Statut final :",
    resultats_pipeline["statut"]
)

Statut final : SUCCÈS


In [136]:
display(
    resultats_pipeline["rapport_qualite"][
        [
            "controle",
            "nombre_anomalies",
            "statut",
            "criticite"
        ]
    ]
)

,controle,nombre_anomalies,statut,criticite
0,Valeurs manquantes dans CLIENT,0,CONFORME,Bloquante
1,Valeurs manquantes dans PRODUIT,0,CONFORME,Bloquante
2,Valeurs manquantes dans PANIER,0,CONFORME,Bloquante
3,Valeurs manquantes dans LIGNE_PANIER,0,CONFORME,Bloquante
4,Doublons complets dans CLIENT,0,CONFORME,Bloquante
5,Doublons complets dans PRODUIT,0,CONFORME,Bloquante
6,Doublons complets dans PANIER,0,CONFORME,Bloquante
7,Doublons complets dans LIGNE_PANIER,0,CONFORME,Bloquante
8,Unicité de client_id,0,CONFORME,Bloquante
9,Unicité de product_id,0,CONFORME,Bloquante


In [137]:
for nom_table, dataframe in (
    resultats_pipeline["tables"].items()
):
    print(
        nom_table,
        dataframe.shape
    )

CLIENT (435, 6)
PRODUIT (200, 4)
PANIER (3333, 4)
LIGNE_PANIER (10343, 6)
